# Post 3 — Newsvendor inventory simulation (§8)

Translates LightGBM quantile forecasts into order quantities via the newsvendor critical ratio, then sums realized stockout + holding cost per regime. TabPFN-TS columns are **pending** until §6.2 is implemented; this notebook produces the LightGBM legs of the §8.4 table and writes `results/newsvendor_costs.parquet`.

> Realism disclaimer (§8.3): observed sales proxy true demand, ignoring stockouts/substitution. Read these as directional model comparisons.

## 0. Control panel

In [ ]:
# ---- Control panel (newsvendor §8) ----------------------------------------
CONFIG_NAME = "default.yaml"
OUTPUT_SUBDIR = "newsvendor"
FEATURE_SETS = ["minimal", "full"]        # §8.4 headline columns
USE_CORE_SLICE = True                     # headline comparison runs on the core slice
TEST_ORIGIN_INDICES = None                # None = all 7; e.g. [3] for the middle origin
SENSITIVITY_CONFIGS = [                   # §8.5 alternative cost ratios
    "sensitivity_perishable_low.yaml", "sensitivity_perishable_high.yaml",
    "sensitivity_nonperishable_low.yaml", "sensitivity_nonperishable_high.yaml",
]
QUICK = True                              # cap LightGBM rounds while iterating
RANDOM_SEED = 42

## 1. Imports

In [ ]:
import sys, time, json, copy, platform, subprocess
from pathlib import Path

# Make `src` importable whether the notebook runs from notebooks/ or repo root.
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src" / "demand").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.demand.config import load_config, resolve_path
from src.demand.data.load import load_raw_data, build_panel
from src.demand.data.splits import (
    build_main_splits, build_core_slice, family_regime, regime_horizon, regime_costs,
)
from src.demand.data.supervised_frame import build_supervised_frame
from src.demand.models import lgbm_point, lgbm_quantile
from src.demand.models.lgbm_base import train_lgbm

def git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"],
                                       stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return "no-git"

## 2. Load data + scope

In [ ]:
cfg = copy.deepcopy(load_config(CONFIG_NAME))
cfg["seed"] = RANDOM_SEED
splits = build_main_splits(cfg)

t0 = time.time()
raw = load_raw_data(cfg)
panel = build_panel(raw, include_test=False)
print(f"loaded panel in {time.time()-t0:.1f}s | rows={len(panel):,} | "
      f"series={panel[['store_nbr','family']].drop_duplicates().shape[0]:,}")

if USE_CORE_SLICE:
    series_filter = build_core_slice(panel, raw.stores, cfg, write_csv=False)
    print(f"core slice: {len(series_filter)} series, "
          f"{series_filter['family'].nunique()} families")
else:
    series_filter = None
    print("scope: full panel")

results_dir = resolve_path(cfg, "results_dir")
figures_dir = resolve_path(cfg, "figures_dir")
out_dir = results_dir / OUTPUT_SUBDIR
out_dir.mkdir(parents=True, exist_ok=True)
print(f"git commit: {git_commit()}  ->  output: {out_dir}")

## 3. Train LightGBM quantile + compute costs

In [ ]:
from src.demand.evaluation.newsvendor import (
    newsvendor_costs, summarize_costs, cost_breakdowns,
)
from src.demand.evaluation import plots

test_origins = list(splits.test_origins)
if TEST_ORIGIN_INDICES is not None:
    test_origins = [test_origins[i] for i in TEST_ORIGIN_INDICES]
train_origins = list(splits.training_origins)
val_origins = list(splits.validation_origins)

def actuals_for(forecasts):
    keys = forecasts[["store_nbr", "family"]].drop_duplicates()
    dates = forecasts["date"].drop_duplicates()
    a = panel.merge(keys, on=["store_nbr", "family"], how="inner")
    return a[a["date"].isin(dates)][["date", "store_nbr", "family", "sales"]]

all_costs = []          # long-format, tagged with approach + feature_set
headline_rows = []      # §8.4 table rows
for fs in FEATURE_SETS:
    print("=" * 70, f"\nfeature_set={fs}")
    # LightGBM quantile triple (p10/p50/p90) — the newsvendor decision needs a
    # distribution, so this is always the quantile variant (§8.4 note).
    frame_train = build_supervised_frame(
        panel, origins=train_origins, horizon=cfg["horizon_days"], feature_set=fs,
        config=cfg, mode="train", holidays=raw.holidays, stores=raw.stores,
        series_filter=series_filter)
    frame_val = build_supervised_frame(
        panel, origins=val_origins, horizon=cfg["horizon_days"], feature_set=fs,
        config=cfg, mode="train", holidays=raw.holidays, stores=raw.stores,
        series_filter=series_filter)
    frame_fc = build_supervised_frame(
        panel, origins=test_origins, horizon=cfg["horizon_days"], feature_set=fs,
        config=cfg, mode="forecast", holidays=raw.holidays, stores=raw.stores,
        series_filter=series_filter)
    if QUICK:
        cfg["lgbm"]["defaults"]["n_estimators"] = 400
    arts = lgbm_quantile.train_alphas(frame_train, frame_val, fs, cfg)
    qfc = lgbm_quantile.predict_quantiles(arts, frame_fc, cfg)   # wide q10/q50/q90
    actuals = actuals_for(qfc)

    costs = newsvendor_costs(qfc, actuals, cfg)
    costs["approach"] = "lgbm_quantile"
    costs["feature_set"] = fs
    all_costs.append(costs)
    summ = summarize_costs(costs)
    print(summ.to_string(index=False))
    for _, r in summ.iterrows():
        headline_rows.append({"approach": "lgbm_quantile", "feature_set": fs,
                              "regime": r["regime"], "cost": r["cost"]})

costs_df = pd.concat(all_costs, ignore_index=True)
costs_df.to_parquet(out_dir / "newsvendor_costs_lgbm.parquet", index=False)
# Canonical artifact consumed by the summary-table generator.
costs_df.to_parquet(results_dir / "newsvendor_costs.parquet", index=False)
headline = pd.DataFrame(headline_rows)
print("\n§8.4 headline (LightGBM legs; TabPFN-TS columns pending §6.2):")
display(headline.pivot_table(index="regime", columns="feature_set", values="cost"))

## 4. Sensitivity to cost ratios (§8.5)

In [ ]:
# §8.5 — re-cost the SAME forecasts under alternative cost ratios. Only the
# critical ratio (hence order quantity) changes; forecasts are untouched.
sens_rows = []
for sc in SENSITIVITY_CONFIGS:
    sc_cfg = copy.deepcopy(load_config(sc))
    for (approach, fs), grp in costs_df.groupby(["approach", "feature_set"]):
        qcols = [c for c in grp.columns if c.startswith("q")]
        actuals = grp[["date", "store_nbr", "family", "actual"]].rename(
            columns={"actual": "sales"})
        recost = newsvendor_costs(grp[["origin", "date", "horizon_offset",
                                       "store_nbr", "family", *qcols]], actuals, sc_cfg)
        s = summarize_costs(recost)
        for _, r in s.iterrows():
            sens_rows.append({"config": sc, "approach": approach, "feature_set": fs,
                              "regime": r["regime"], "cost": r["cost"]})
sens_df = pd.DataFrame(sens_rows)
sens_df.to_parquet(results_dir / "newsvendor_sensitivity.parquet", index=False)
display(sens_df.pivot_table(index=["config", "regime"], columns="feature_set", values="cost"))

## 5. Breakdown charts + calibration (§8.6, §7.3)

In [ ]:
# §8.6 breakdown charts + calibration sanity check.
breakdowns = cost_breakdowns(costs_df)
plots.newsvendor_bar(headline.rename(columns={"approach": "approach"}), figures_dir / OUTPUT_SUBDIR)
plots.cost_by_family(breakdowns["by_family"], figures_dir / OUTPUT_SUBDIR)
plots.cumulative_cost(breakdowns["cumulative_by_date"].assign(approach="lgbm_quantile"),
                      figures_dir / OUTPUT_SUBDIR)

# Calibration (§7.3): one plot per feature set, p10/p50/p90 vs empirical coverage.
for fs in FEATURE_SETS:
    g = costs_df[costs_df["feature_set"] == fs]
    if {"q10", "q50", "q90"}.issubset(g.columns) and not g.empty:
        plots.calibration_plot(g["actual"].to_numpy(), g["q10"].to_numpy(),
                               g["q50"].to_numpy(), g["q90"].to_numpy(),
                               figures_dir / OUTPUT_SUBDIR, f"calibration_{fs}.png")
print("figures ->", figures_dir / OUTPUT_SUBDIR)

# Refresh the headline markdown so the §8.4 table reflects this run.
from src.demand.reporting.summary_tables import build_summary_tables
print("summary ->", build_summary_tables(cfg))

## 6. TabPFN-TS leg — _pending_

When §6.2 lands, sample p10/p50/p90 from TabPFN-TS native output on the same `(series, origin, horizon)` rows, tag with `approach="tabpfn_ts"`, append to `newsvendor_costs.parquet`, and re-run §3–5. No other changes.